# 01 · export v2 production log — 네 조각 + canonical log

**Kernel: `fttl-v2` (env-v2).** v2 repo의 `lvanalytics/evaluation_helpers.py`로 프로덕션 로그를 읽고,
v3 extract의 `Fttl`을 붙여 outcome을 만든 뒤 parquet으로 내보낸다. 로그 프레임은 v2 쪽에 묶여
있으므로 여기(env-v2)서 parquet으로 떨어뜨려야 분석 env가 읽는다.

`v2_logs` 는 **한 종류의 데이터가 아니다.** raw ↔ transformed 를 `ClaimNumber` + `correlation_id` 로
`how="inner"`, `suffix="_transformed"` join 하고 거기에 prediction 을 붙인 프레임이다. 그대로
내보내면 어느 컬럼이 raw 이고 어느 것이 모델이 실제로 먹은 값인지 파일에서 구분되지 않으므로,
**컬럼으로 네 조각을 내서** 쓴다.

- **키는 두 개다**: `ClaimNumber` + `correlation_id`. 한 클레임이 두 번 이상 스코어될 수 있다.
- **접미사는 이름이 겹친 컬럼에만 붙어 있다** — 나누는 기준은 `param.py` 의 `MODEL_FEATURES`.
- score = **`FastTrackerProbablity`** (repo 철자 그대로), decision = **`FastTrackerDecision`**
- observed = v3 extract의 **`Fttl`** — 로그 자체에는 outcome이 없다

| writes | kind | 컬럼 |
|---|---|---|
| `logs/v2_raw.parquet` | `log_raw` | 키 + 나머지 raw 컬럼 (v2 이름) |
| `logs/v2_features.parquet` | `log_features` | 키 + MODEL_FEATURES (fit 순서, 접미사 뗀 이름) |
| `logs/v2_scores.parquet` | `log_scores` | 키 + `score`, `decision` |
| `logs/v2_targets.parquet` | `log_targets` | 키 + `date`, `observed` |
| `logs/v2.parquet` | `log` | **클레임당 한 행**: `claim_id, date, score, decision, observed` |

경로는 전부 `config.path(kind, "v2", "real")`에서 나온다 — 파일명을 노트북에 박지 않는다.
`logs/v2.parquet`은 지금 **더미**가 들어 있고, 이 노트북이 그 자리를 진짜 로그로 덮어쓴다.
이 canonical log 는 `loaders.load("v2").log` 가 읽는 형식이다.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import config                      # noqa: E402

# 커널 확인: 분석용 .venv가 아니라 env-v2여야 한다 (두 줄이 같은 인터프리터인지 눈으로 확인)
print("kernel :", sys.executable)
print("env-v2 :", config.python_bin("v2"))


In [ ]:
# lvanalytics/ 를 담고 있는 디렉터리만 sys.path 에 넣으면 평범한 import 문이 통한다.
# repo 루트 바로 아래가 아니면 여기만 실제 부모로 바꾼다 (예: config.repo("v2") / "src").
sys.path.insert(0, str(config.repo("v2")))

from lvanalytics import evaluation_helpers as eh   # noqa: E402

print(eh.__file__)
print([n for n in dir(eh) if not n.startswith("_")])


---
## v2_logs 안에서 컬럼으로 쪼갠다

`v2_logs` 한 프레임 안에 raw · transformed · prediction 이 다 들어 있다 (raw ↔ transformed 를
`ClaimNumber` + `correlation_id` 로 `how="inner"`, `suffix="_transformed"` join 한 뒤 prediction 을
붙인 것). 프레임이 하나뿐이므로 행으로 나눌 것은 없고, **컬럼을 어느 쪽 것인지 판정**해서 쪼갠다.

extract 에서 가져오는 것은 `Fttl` 하나뿐이다. 키 컬럼은 extract 쪽 이름(`Claimnumber_CLAIM`)을
v2 로그 이름으로 바꿔서 붙인다 — 값 형식은 양쪽이 같다.

In [ ]:
# ---- SOURCE: 프레임 하나 + 키 -------------------------------------------------------
import polars as pl                        # noqa: E402

LOG = v2_logs                              # raw + transformed + pred 가 다 들어 있는 프레임
EXTRACT = extract                          # v3 extract (Fttl 을 가져올 곳)

LOG_ID = "ClaimNumber"                     # 로그 쪽 claim number
EVENT_ID = "correlation_id"                # 스코어링 이벤트 id — 이게 있어야 한 행이 유일하다
KEYS = [LOG_ID, EVENT_ID]

EXT_ID = "Claimnumber_CLAIM"               # extract 쪽
OBSERVED = "Fttl"                          # extract 쪽 라벨 = config.column('v3','observed')
SUFFIX = "_transformed"                    # join 이 붙인 접미사 — 겹친 이름에만 붙어 있다

miss = [k for k in KEYS if k not in LOG.columns]
assert not miss, (miss, [c for c in LOG.columns
                         if "claim" in c.lower() or "corr" in c.lower()])
dup = LOG.height - LOG.select(KEYS).unique().height
assert dup == 0, f"(claim, correlation) 이 유일하지 않다 — 키가 더 있다 ({dup} 건)"
print("LOG    ", LOG.shape, " 키 중복", dup)
print("extract", EXTRACT.shape)

In [ ]:
# ---- MODEL_FEATURES: 컬럼을 나누는 기준 ------------------------------------------------
# v2 repo 가 이미 sys.path 에 있으므로 param.py 를 그대로 import 한다 (붙여넣은 리스트는 repo 가
# 바뀌면 조용히 낡는다). fit 순서는 env-v2 안에서만 알 수 있으므로 파일이 그 순서를 들고 나간다.
try:
    from param import MODEL_FEATURES      # noqa: E402
    SRC = "param.py (import)"
except ImportError as exc:
    print("param.py import 실패:", exc, "— 아래 리스트에 param.py 내용을 그대로 붙여넣을 것")
    MODEL_FEATURES = [
        # <<< paste v2 param.py MODEL_FEATURES here, in its original order >>>
    ]
    SRC = "pasted"

assert MODEL_FEATURES, "MODEL_FEATURES 가 비어 있다"
dupes = sorted({c for c in MODEL_FEATURES if MODEL_FEATURES.count(c) > 1})
assert not dupes, f"param 리스트에 중복: {dupes}"
print(f"MODEL_FEATURES {len(MODEL_FEATURES)}  ({SRC})   ·   LOG {LOG.width} columns")

In [ ]:
# ---- 컬럼 쪼개기: 어느 컬럼이 trans / pred / raw 인가 ----------------------------------
# suffix 는 **이름이 겹친 컬럼에만** 붙는다. 그래서 접미사만 보고 나누면 겹치지 않은 transformed
# 컬럼이 raw 로 새어 나간다. MODEL_FEATURES 를 기준으로 하나씩 판정한다:
#   f + SUFFIX 가 있으면  -> 그게 trans, 맨이름 f 는 raw (겹쳐서 붙은 경우)
#   f 만 있으면           -> 겹치지 않았으므로 그 f 가 trans
import config                               # noqa: E402

SCORE = config.column("v2", "score")           # "FastTrackerProbablity"
DECISION = config.column("v2", "decision")     # 로그의 fast-track 플래그
DATE = config.column("v2", "date")             # "ReportedDate"
PRED_COLS = [SCORE, DECISION]

cols = set(LOG.columns)
# decision 이름은 아직 로그로 확인된 적이 없다 — 없으면 후보를 찍고 멈춘다.
assert DECISION in cols, [c for c in LOG.columns
                          if "track" in c.lower() or "decis" in c.lower()]
assert SCORE in cols, [c for c in LOG.columns if "prob" in c.lower()]
assert DATE in cols, [c for c in LOG.columns if "date" in c.lower()]

TRANS_MAP, missing, collided = {}, [], []     # LOG 의 컬럼 -> 모델이 아는 이름 (fit 순서)
for f in MODEL_FEATURES:
    if f + SUFFIX in cols:
        TRANS_MAP[f + SUFFIX] = f
        collided.append(f)
    elif f in cols:
        TRANS_MAP[f] = f
    else:
        missing.append(f)

RAW_COLS = [c for c in LOG.columns
            if c not in TRANS_MAP and c not in PRED_COLS and c not in KEYS]
leftover = [c for c in RAW_COLS if c.endswith(SUFFIX)]   # MODEL_FEATURES 로 설명 안 되는 접미사

# join 전 프레임이 세션에 남아 있으면 위 판정을 추론이 아니라 사실로 확인한다. TRANS_MAP 이
# raw 컬럼을 먹는 경우는 "MODEL_FEATURES 에 있는데 transformed 프레임에는 없는 이름" 하나뿐이고,
# 그건 여기서만 잡힌다 (합쳐진 프레임만 보면 겹치지 않은 trans 컬럼과 구분이 안 된다).
if "v2_transformed_logs" in dir():
    real_trans = set(v2_transformed_logs.columns)
    stolen = [f for src, f in TRANS_MAP.items() if src == f and f not in real_trans]
    assert not stolen, f"raw 컬럼을 trans 로 잘못 분류: {stolen[:20]}"
    print(f"  ✓ v2_transformed_logs 로 검증 ({len(real_trans)} cols)")
else:
    print("  · join 전 프레임(v2_transformed_logs)이 없어 suffix 규칙으로 추론함")

print(f"  trans  {len(TRANS_MAP):>4}  (그 중 이름이 겹쳐 접미사가 붙은 것 {len(collided)})")
print(f"  raw    {len(RAW_COLS):>4}")
print(f"  pred   {len(PRED_COLS):>4}  {PRED_COLS}")
print(f"  MODEL_FEATURES 인데 LOG 에 없음 {len(missing)}  {missing[:20]}")
print(f"  _transformed 인데 MODEL_FEATURES 가 아님 {len(leftover)}  {leftover[:20]}")
# 위 두 줄이 0 이 아니면 그대로 넘어가지 말 것 — 빠진 feature 는 파일에도 없고,
# 설명 안 되는 _transformed 컬럼은 raw 쪽으로 새어 들어간다.

# 겹친 feature 는 같은 이름이 두 파일에 한 번씩 나간다: raw 쪽 값은 log_raw 에, 변환된 값은
# log_features 에. 값이 실제로 다른지 (= 전처리가 그 컬럼에 무엇을 했는지) 눈으로 확인한다.
if collided:
    ex = collided[:3]
    print(LOG.select([c for f in ex for c in (f, f + SUFFIX)]).head(3))

In [ ]:
# ---- 진단: MODEL_FEATURES 인데 LOG 에 없는 이름 --------------------------------------
# missing 은 "param.py 의 이름이 LOG 에 맨이름으로도 접미사로도 없다" 는 뜻이다. 후보는 셋:
#   (a) param.py 가 배포 이후 바뀌었다  (b) 다른 param.py 를 import 했다  (c) 로그가 안 남긴다
# 판정 기준은 param.py 가 아니라 **배포된 model.pkl 이 아는 이름**이다 — 그게 fit 순서의 출처다.
import re                                   # noqa: E402
import joblib                               # noqa: E402
import param                                # noqa: E402

# 1) 어느 param.py 를 읽었나 — 같은 이름 파일이 sys.path 에 둘 이상이면 조용히 딴 것을 먹는다
print("param.py :", param.__file__)

# 2) 빠진 이름 전부 + LOG 쪽 비슷한 이름 (대소문자/구분자만 다른 경우를 잡는다)
def _n(s): return re.sub(r"[^a-z0-9]", "", s.lower())
by_norm = {}
for c in LOG.columns:
    by_norm.setdefault(_n(c[: -len(SUFFIX)] if c.endswith(SUFFIX) else c), []).append(c)
print(f"\nmissing {len(missing)}: {missing}")
for f in missing:
    near = by_norm.get(_n(f)) or [c for c in LOG.columns if _n(f)[:6] in _n(c)]
    print(f"  {f:<42} LOG 후보 {near[:6]}")
    if "v2_transformed_logs" in dir():
        print(f"  {'':<42} transformed 프레임에 있나: {f in v2_transformed_logs.columns}")

# 3) 배포된 모델 자신이 아는 이름과 대조 — param.py 드리프트면 여기서 갈린다
try:
    _m = joblib.load(config.path("model", "v2", "real"))
    FIT_NAMES = list(getattr(_m, "feature_names_in_", None) or _m.get_booster().feature_names)
    print(f"\nmodel.pkl feature_names {len(FIT_NAMES)}  vs  param {len(MODEL_FEATURES)}")
    print("  param 에만 있음      :", [f for f in MODEL_FEATURES if f not in FIT_NAMES])
    print("  model 에만 있음      :", [f for f in FIT_NAMES if f not in MODEL_FEATURES])
    print("  순서까지 같나         :", FIT_NAMES == list(MODEL_FEATURES))
    print("  missing 중 model 도 모름:", [f for f in missing if f not in FIT_NAMES])
except Exception as exc:                     # noqa: BLE001
    print("\nmodel.pkl 못 읽음:", type(exc).__name__, exc)

# 판정:
#   missing 이 전부 "model 도 모름"  -> param.py 가 배포 이후 바뀐 것. FIT_NAMES 를 기준으로
#       쓰면 missing 은 0 이 된다 (아래 한 줄 켜고 셀 6 을 다시 실행).
#           # MODEL_FEATURES = FIT_NAMES
#   missing 이 model 에는 있다      -> 로그가 그 컬럼을 안 남긴 것. log_features 는 모델 입력을
#       완전히 재현하지 못한다 — 채우지 말고 이 사실을 기록하고, SHAP 은 이 3 개를 뺀 집합으로.

# 4) 접미사가 어느 쪽에 붙었는지 값으로 확인 (join 의 left/right 가 반대면 raw 를 feature 로 쓴다)
if collided and "v2_transformed_logs" in dir():
    f = collided[0]
    t = v2_transformed_logs.select(pl.col(f).alias("trans_frame")).head(5)
    print("\n접미사 방향 확인 —", f)
    print(pl.concat([LOG.select([f, f + SUFFIX]).head(5), t], how="horizontal"))


In [ ]:
# ---- missing 3 개는 전부 one-hot 더미다: 부모 카테고리를 raw 쪽에서 센다 ---------------
# LOG 어디에도 (맨이름·접미사 둘 다) 없다는 것까지는 확인됐다. 남은 갈림길은 하나다:
#   부모 레벨이 로그 창에 0 건   -> 인코더가 컬럼을 아예 안 만든 것. 그 컬럼은 원래 전부 0 이라
#                                  0 으로 채우는 것이 transform 이 냈을 값과 **같다**.
#   부모 레벨이 있는데 컬럼 없음 -> 로그가 모델 입력을 다 안 남긴 것. 채우면 안 된다.
# one-hot 은 문자열 완전 일치로만 켜지므로 Alloy / Alloys 같은 철자 갈림도 여기서 드러난다.
for f in missing:
    stem = f.split("_")[0]
    par = [c for c in LOG.columns if c.lower().startswith(stem.lower()[:10])]
    print(f"\n{f}\n  부모 후보: {par[:8]}")
    for c in par[:3]:
        s = LOG[c]
        if s.dtype == pl.Utf8 or s.n_unique() <= 30:
            print(s.value_counts(sort=True).head(12))


In [ ]:
# ---- 재현: pipeline.pkl 을 log_raw 에 직접 돌린다 ---------------------------------------
# 로그의 transformed 컬럼을 주워 모으는 대신, 프로덕션이 쓴 그 전처리기를 raw 에 그대로 먹인다.
# 나오는 행렬이 모델 입력의 원본이고 — missing 3 개도 여기서는 나온다 (레벨이 0 건이면 전부 0).
# 재현한 점수가 로그의 score 와 맞으면 log_features 전체의 출처가 한 번에 검증된다.
import numpy as np                          # noqa: E402
import joblib                               # noqa: E402

pipe = joblib.load(config.path("preprocessor", "v2", "real"))
raw_pd = LOG.select(RAW_COLS).to_pandas()          # collided feature 의 맨이름 = raw 쪽
X = pipe.transform(raw_pd)
print("transform out", X.shape)
print("  MODEL_FEATURES 중 여전히 없음:", [f for f in MODEL_FEATURES if f not in X.columns])
print("  missing 3 개의 값 분포:")
for f in missing:
    if f in X.columns:
        print(f"    {f:<45} sum={X[f].sum()}  nunique={X[f].nunique()}")

est = joblib.load(config.path("model", "v2", "real"))
p = est.predict_proba(X[list(MODEL_FEATURES)])[:, 1]
diff = np.abs(p - LOG[SCORE].to_numpy())
print(f"\nscore 재현  |diff| max {diff.max():.3e}   median {np.median(diff):.3e}")
# max 가 1e-6 근처면 재현 성공 -> TRANS_MAP 대신 X 를 log_features 로 내보낸다 (fit 순서 그대로):
#     log_features = pl.concat([LOG.select(KEY_OUT), pl.from_pandas(X[list(MODEL_FEATURES)])],
#                              how="horizontal")
# 안 맞으면 프로덕션 전처리가 지금 repo 의 pipeline.pkl 과 다르다는 뜻이고, 그건 채워 넣을 게
# 아니라 기록할 사실이다.


In [ ]:
# ---- join: extract 에서 Fttl 만 붙인다 ------------------------------------------------
right = (EXTRACT.select([pl.col(EXT_ID).alias(LOG_ID), pl.col(OBSERVED)])
                .unique(subset=[LOG_ID], keep="first"))   # 중복이 있으면 join 이 행을 늘린다

joined = LOG.join(right, on=LOG_ID, how="left")           # inner 면 매칭 안 된 로그 행이 사라진다
assert joined.height == LOG.height, (joined.height, LOG.height)
print(f"join coverage {joined[OBSERVED].is_not_null().mean():.1%}"
      f"  ({joined[OBSERVED].is_not_null().sum()} / {joined.height})")

In [ ]:
# ---- 네 조각 + log -------------------------------------------------------------------
import schema                               # noqa: E402

# 키는 네 조각 모두 같은 이름으로 나간다: claim number 는 canonical claim_id, correlation_id 는
# canonical 이름이 없으므로 원래 이름 그대로. 키만 통일하면 조각들이 항상 같은 식으로 다시 붙는다.
KEY_OUT = [pl.col(LOG_ID).alias(schema.CLAIM_ID), pl.col(EVENT_ID)]

# 값 컬럼은 다르다: raw / features 는 v2 이름 그대로 (feature 이름 자체가 측정값이다),
# score / targets 는 canonical. features 는 접미사를 떼고 **모델이 아는 이름**으로 되돌린다
# — SHAP·registry·feature_overlap 이 전부 그 이름으로 붙는다.
log_raw = LOG.select([*KEY_OUT, *RAW_COLS])
log_features = LOG.select([*KEY_OUT, *TRANS_MAP.keys()]).rename(TRANS_MAP)
log_scores = joined.select([
    *KEY_OUT,
    pl.col(SCORE).alias(schema.SCORE),
    pl.col(DECISION).alias(schema.DECISION),
])
log_targets = joined.select([
    *KEY_OUT,
    pl.col(DATE).alias(schema.DATE),
    pl.col(OBSERVED).alias(schema.OBSERVED),
])

log = (log_scores.join(log_targets, on=[schema.CLAIM_ID, EVENT_ID], how="left")
                 .select([schema.CLAIM_ID, schema.DATE, schema.SCORE,
                          schema.DECISION, schema.OBSERVED]))

# log 만 클레임당 한 행이어야 한다 (loaders / threshold.read_off 가 그렇게 읽는다). 한 클레임에
# 이벤트가 여럿이면 어느 이벤트가 그 클레임의 결정인지는 분석 선택이므로 조용히 고르지 않는다.
n_dup = log.height - log[schema.CLAIM_ID].n_unique()
assert n_dup == 0, (f"클레임 {n_dup} 건이 두 번 이상 스코어됐다 — 어느 이벤트를 쓸지 정할 것 "
                    f"(예: 시각 컬럼으로 sort 후 unique(subset=[claim_id], keep='first'))")
print("log", log.shape, "· features", log_features.shape, "· raw", log_raw.shape)
print("scrap rate", f"{log[schema.DECISION].mean():.4f}",
      "· observed pos rate", f"{log[schema.OBSERVED].mean():.4f} (null 제외)")

In [ ]:
# ---- write: 경로는 config 의 kind 에서 -----------------------------------------------
WRITTEN = []
for df, kind in ((log_raw, "log_raw"), (log_features, "log_features"),
                 (log_scores, "log_scores"), (log_targets, "log_targets"), (log, "log")):
    dst = config.path(kind, "v2", "real")
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        print(f"OVERWRITING {dst.name} — 지금까지 들어 있던 것은 make_dummy_real.py 의 더미")
    df.write_parquet(dst)
    WRITTEN.append((dst, df.height, df.columns))
    print(f"{kind:<13} -> {dst.relative_to(config.ROOT)}  rows {df.height}  cols {df.width}")

메모:

- 산출물은 전부 **kind** 로 잡혀 있다: `log` 는 원래 있던 kind(`loaders.load("v2").log`),
  `log_raw` / `log_features` / `log_scores` / `log_targets` 는 이 노트북 때문에 있는 kind다.
  파일명·디렉터리는 `config.FALLBACK` 한 줄에만 있고 노트북에는 없다.
- **소스는 `v2_logs` 하나다.** 네 조각은 같은 행 집합(이벤트당 한 행)을 컬럼으로 나눈 것이고,
  키 `[claim_id, correlation_id]` 로 언제든 다시 붙는다. `log` 만 클레임당 한 행이다.
- **접미사만 보고 나누지 말 것.** `_transformed` 는 이름이 겹친 컬럼에만 붙어 있어서, 겹치지 않은
  transformed 컬럼은 맨이름으로 들어와 있다. 판정 기준은 `MODEL_FEATURES` 다.
- `log_features` 의 컬럼은 **접미사를 뗀 모델의 이름**이다. `feature_x_transformed` 로 내보내면
  SHAP·registry·feature_overlap 이 전부 그 이름을 못 찾는다.
- **키는 네 조각 모두 `claim_id` + `correlation_id`.** `correlation_id` 는 canonical 이름이 없어서
  원래 이름 그대로 나간다. 값 컬럼만 `log_raw` / `log_features` 가 v2 이름, 나머지가 canonical 이다.
  feature 이름은 측정값 자체라 절대 바꾸지 않고, score/decision/date/observed 는 구조 컬럼이라
  `schema` 이름으로 나간다.
- `log_features` 는 **split kind 가 아니다.** 학습 split 의 `features_v2_{split}.parquet` 은 Z:
  transformed 프레임에서 나온 행이고, 이쪽은 프로덕션이 실제로 스코어한 행이다. 다른 행 집합이므로
  섞지 말 것.
- join 은 left 로 둔다. inner 면 extract 에 없는 로그 행이 사라져 로그 행 수가 바뀐다.
- `observed` 는 v3 extract 의 `Fttl` 이고, extract 창(2023-06~2026-05) 밖의 로그 행은 null 이다.
  이 null 은 "라벨 미관측"이지 0 이 아니다 — 절대 fillna(0) 하지 말 것.
- `decision` 은 로그가 들고 있는 **실제 fast-track 플래그**여야 한다. `score >= τ` 로 만들어
  쓰지 말 것 — v2 의 τ 는 기간에 따라 다섯 번 바뀌었으므로 단일 τ 로 재구성하면 틀린다.